# <center>**Enhancing LEM-X Imaging with the IROS Reconstruction Pipeline**<center>

## <center>**Sky Reconstruction Efficiency**<center>

In [1]:
from pathlib import Path
from typing import Any, Callable

import numpy as np
import pandas as pd

from bloodmoon.mask import CodedMaskCamera, codedmask
from bloodmoon.io import simulation_files
from bloodmoon.types import CoordEquatorial
import darksun as ds
from darksun.data import Log, DataLoader, CatalogueLoader

from IROSrec.handle import config_dirpaths
import imgmaker as mgm
from imgmaker.fns import CameraUnitMap

In [2]:
MASK_FITS: str = "mask_NTHT_20260129_CORRECTED.fits"

SKYFIELD: str = "GalacticCentre"
# SKYFIELD: str = "IROSDummy"
DATA_FITS: str = "baseline_2-50keV_1ks"

RUN_ID: str = 'GC_rec_yElsHidden'

ID_CAMERA_A: str = "cam1a"
ID_CAMERA_B: str = "cam1b"
DATASET: str = "reconstructed"

E_min: float = 2.0  # [keV]
E_max: float = 50.0  # [keV]
coords2exclude: list[CoordEquatorial] | None = None

UP_X, UP_Y = 2, 1
hide_bulk_els_y: float = 1.5   # [mm]

In [3]:
MASK_PATH, SIMUL_DATA_PATH, SAVE_PATH = config_dirpaths(
    mask=MASK_FITS,
    skyfield=SKYFIELD,
    simul=DATA_FITS,
    runID=RUN_ID,
)
OUT_RESULTS_PATH = mgm.config_savedata_to()

wfm: CodedMaskCamera = codedmask(MASK_PATH, UP_X, UP_Y, hide_bulk_els_y=hide_bulk_els_y)

filepaths: dict[str, dict[str, Path]] = simulation_files(SIMUL_DATA_PATH)
sdlA = ds.get_data(filepaths[ID_CAMERA_A][DATASET], E_min=E_min, E_max=E_max, coords=coords2exclude)
catA = ds.get_catalogue(filepaths[ID_CAMERA_A]['sources'])
sdlB = ds.get_data(filepaths[ID_CAMERA_B][DATASET], E_min=E_min, E_max=E_max, coords=coords2exclude)
catB = ds.get_catalogue(filepaths[ID_CAMERA_B]['sources'])

logA, logB = ds.load_database(f"{SAVE_PATH}/IROS_sources_db.fits")

# Loading data...
# Loading completed!


### <center>**Benchmark Tables**<center>

In [4]:
import re

def adjust_Tabfrmt(txt: str) -> str:
    # insert \hline instead of rules (journal guidelines)
    for rule in ('toprule', 'midrule', 'bottomrule'):
        txt = txt.replace(rule, 'hline')
    # shift caption and label at the end (journal guidelines)
    pattern = r"(\\begin\{table\}.*?)(\\caption\{.*?\})\s*(\\label\{.*?\})\s*(\\begin\{tabular\}.*?\\end\{tabular\})"
    replacement = r"\1\4\n\2\n\3"
    txt = re.sub(pattern, replacement, txt, flags=re.DOTALL)
    # convert to onecolumn
    txt = txt.replace('table', 'table*')
    return txt

def sort_by(df: pd.DataFrame, key: str, **kwargs: Any) -> pd.DataFrame:
    """Sort DataFrame wrt input column key."""
    return df.sort_values(by=[key], ascending=False, ignore_index=True, **kwargs)

In [5]:
from numpy.typing import NDArray

def comp_fc_mstd(varmap: NDArray, boxsize: tuple[int, int]) -> float:
    """Computes the RMSE of the given varmap in specified array box."""
    n, m = varmap.shape
    srows, scols = (
        slice((n - 1) // 2 - boxsize[0], (n - 1) // 2 + boxsize[0] + 1),
        slice((m - 1) // 2 - boxsize[1], (m - 1) // 2 + boxsize[1] + 1),
    )
    mstd = np.sqrt(np.mean(varmap[srows, scols]))
    return mstd

def comp_src_mstd(log: Log, varmap: NDArray, boxsize: tuple[int, int]) -> NDArray:
    """Computes the RMSE for each IROS source from given varmap in specified array box."""
    mstds: list[float] = []
    boxsize_ = (max(boxsize[0], 1), max(boxsize[1], 1))
    for y, x in zip(log.log['y'], log.log['x']):
        srows, scols = (
            slice(y - boxsize_[0], y + boxsize_[0] + 1),
            slice(x - boxsize_[1], x + boxsize_[1] + 1),
        )
        mstd = np.sqrt(np.mean(varmap[srows, scols]))
        mstds.append(mstd)
    return np.array(mstds)

def gather_cam_data(
    log: Log,
    catalogue: CatalogueLoader,
    sdl: DataLoader,
    camera: CodedMaskCamera,
    varmap: NDArray,
) -> pd.DataFrame:
    """
    Gathers single camera data from IROS reconstruction database.
    """
    ids = np.array([src.upper() for src in log.log['ID']])
    theta_res_x, theta_res_y = mgm.get_angularcoords_residues(log, catalogue, sdl, camera)
    cts = np.array(log.log['fluence'])
    true_cts = mgm.extract_catalogue_fluences(log, catalogue, sdl, camera)
    # rmse = comp_fc_mstd(varmap, boxsize=(camera.upscale_f.y * 80, camera.upscale_f.x * 200))
    src_mstd = comp_src_mstd(log, varmap, boxsize=tuple(int(np.ceil(a // 2)) for a in ds.psf_extension(camera)[::-1]))
    dmap = {
        log.name: {
            'Source': ids,
            'DthetaX': theta_res_x,
            'DthetaY': theta_res_y,
            'IROS_cts': cts,
            'True_cts': true_cts,
            # 'Dcts': (cts - true_cts) / np.sqrt(true_cts),
            'Dcts': (cts - true_cts) / src_mstd,
            # 'Dcts_var': (cts - true_cts) / rmse,
            'SNR': np.array(log.log['snr']),
            # 'thetaX [deg]': np.array(log.log['angle_x']),
            # 'thetaY [deg]': np.array(log.log['angle_y']),
        }
    }
    return pd.DataFrame(dmap)

def get_joint_tab(
    data_camA: pd.DataFrame,
    data_camB: pd.DataFrame,
    unitmap: CameraUnitMap,
) -> pd.DataFrame:
    """Generates a Dataframe with output data from both cameras."""
    compose: Callable = lambda a, b: np.sqrt(a ** 2 + b ** 2)
    dmap = {
        'Source': np.array(data_camA.CAM1A['Source'])[unitmap.idx_a],

        'DthetaX_A': np.array(data_camA.CAM1A['DthetaX'])[unitmap.idx_a],
        'DthetaY_A': np.array(data_camA.CAM1A['DthetaY'])[unitmap.idx_a],
        'TrueCts_A': np.array(data_camA.CAM1A['True_cts'])[unitmap.idx_a],
        'ReconstrCts_A': np.array(data_camA.CAM1A['IROS_cts'])[unitmap.idx_a],
        'Dcts_A': np.array(data_camA.CAM1A['Dcts'])[unitmap.idx_a],
        # 'Dcts_A_var': np.array(data_camA.CAM1A['Dcts_var'])[unitmap.idx_a],

        'DthetaX_B': np.array(data_camB.CAM1B['DthetaX'])[unitmap.idx_b],
        'DthetaY_B': np.array(data_camB.CAM1B['DthetaY'])[unitmap.idx_b],
        'TrueCts_B': np.array(data_camB.CAM1B['True_cts'])[unitmap.idx_b],
        'ReconstrCts_B': np.array(data_camB.CAM1B['IROS_cts'])[unitmap.idx_b],
        'Dcts_B': np.array(data_camB.CAM1B['Dcts'])[unitmap.idx_b],
        # 'Dcts_B_var': np.array(data_camB.CAM1B['Dcts_var'])[unitmap.idx_b],

        'SNR': compose(
            np.array(data_camA.CAM1A['SNR'])[unitmap.idx_a],
            np.array(data_camB.CAM1B['SNR'])[unitmap.idx_b],
        ),
    }
    return pd.DataFrame(dmap)

In [6]:
from bloodmoon.mask import count, variance

def get_varmap(camera: CodedMaskCamera, sdl: DataLoader) -> NDArray:
    detector = count(camera, sdl.DLdata)[0]
    varmap = variance(camera, detector)
    return varmap


varmapA, varmapB = map(lambda sdl: get_varmap(wfm, sdl), (sdlA, sdlB))

UserInfo: using bulk mask of [0.0 x 1.5] mm.


In [7]:
ds.pixels_angular_resolution(wfm)
cu_map = mgm.get_srcmap_for_unit(logA.log['ID'], logB.log['ID'])

# Table - CAMERA A
data_camA = gather_cam_data(logA, catA, sdlA, wfm, varmapA)

# Table - CAMERA B
data_camB = gather_cam_data(logB, catB, sdlB, wfm, varmapB)


Pixel angular resolution at upscaling (x, y): (2, 1)
  - fine direction: 2.1163 arcmin
  - coarse direction: 8.4653 arcmin



Analysing SCOX1:   0%|          | 0/25 [00:00<?, ?it/s]WARNING: The following header keyword is invalid or follows an unrecognized non-standard convention:
CATALOG =RXTE-ASM_BeppoSAX-WFC_catalog_2-50keV.fits / Catalog file               [astropy.io.fits.card]
Analysing SCOX1:   0%|          | 0/21 [00:00<?, ?it/s]WARNING: The following header keyword is invalid or follows an unrecognized non-standard convention:
CATALOG =RXTE-ASM_BeppoSAX-WFC_catalog_2-50keV.fits / Catalog file               [astropy.io.fits.card]
Analysing LEMX-CAM1BS2: 100%|██████████| 21/21 [00:02<00:00,  8.07it/s]


In [8]:
unit_data = get_joint_tab(data_camA, data_camB, cu_map)

KWS = {
    'label': 'Table1',
    'caption': 'Testing $`to\\_latex`$ fn.',
    'float_format': "%.4f",
    'column_format': 'l' + 'c' * (len(unit_data.columns) - 2) + 'r',
}
tab = mgm.df2TeXtab(
    df=sort_by(unit_data, 'SNR'),
    adjust_tabfrmt=adjust_Tabfrmt,
    # save_to=f'{OUT_RESULTS_PATH}/../texTable_Unit_results_{DATASET}_{E_min}-{E_max}.tex',
    overwrite=True,
    **KWS,
)

In [9]:
unit_data.sort_values('SNR', ascending=False, ignore_index=True)

,Source,DthetaX_A,DthetaY_A,TrueCts_A,ReconstrCts_A,Dcts_A,DthetaX_B,DthetaY_B,TrueCts_B,ReconstrCts_B,Dcts_B,SNR
0,SCOX1,0.090131,0.396306,988467.0,978664.126519,-7.945514,0.153977,-0.041911,882594.0,863127.169448,-16.580119,968.440508
1,GX5-1,-0.026234,-1.235176,108098.0,106236.887629,-1.388974,-0.033027,-1.495042,109186.0,110329.296738,0.870158,102.851922
2,GX349+2,0.049448,0.713051,77561.0,79005.623395,1.077677,0.010332,1.444055,78173.0,79731.063920,1.185546,67.560457
3,GX9+1,0.126945,1.129088,59848.0,62349.559381,1.866078,0.023681,1.855954,60359.0,61680.980430,1.006115,58.343059
4,GX17+2,0.237318,-4.169966,67943.0,66436.773350,-1.134493,-0.018877,-2.600004,72053.0,71352.419252,-0.533817,56.578067
5,GX340+0,-0.212867,6.510455,43244.0,39617.459312,-2.852174,-0.004480,1.186508,46651.0,49790.838103,2.457712,42.779685
6,GX13+1,0.178867,-1.574253,34396.0,38992.547416,3.428912,-0.022485,-1.912514,34730.0,39736.150700,3.809149,37.017282
7,X1820-303,-0.169990,7.116843,32712.0,37800.314095,3.795981,-0.174925,-4.541084,32027.0,32925.047594,0.683344,31.329011
8,GX3+1,0.041634,3.527372,33917.0,38988.659444,3.784150,0.178813,-3.239409,34641.0,34370.182755,-0.206091,30.500893
9,CIRX1,-0.622346,8.230652,19911.0,18754.301851,-1.355824,0.326312,2.258315,21248.0,21270.381861,0.027592,30.182396


In [12]:
logB.log['ID']
# logA.log['fluence']

['scox1',
 'gx5-1',
 'gx349+2',
 'gx9+1',
 'gx17+2',
 'gx340+0',
 'gx13+1',
 'x1820-303',
 'gx3+1',
 'gx9+9',
 'groj1655-40',
 'cirx1',
 'x1735-444',
 'x1705-440',
 'serx1',
 'gx354-0',
 'gx339-4',
 'grs1915+105',
 'x1624-490',
 'lemx-cam1bS1',
 'lemx-cam1bS2']